# Modelado Relacional con SQLite
### Fundamentos de Gestión de Datos · TECSUP · Semana 4 · Laboratorio Dirigido D4

**Docente:** Pilar Rocío Sayán Mejía · **Motor:** SQLite · **Entorno:** Google Colab (Python)

| | |
|---|---|
| **Tema** | Entidad-Relación, cardinalidad y normalización |
| **Requisito** | Haber terminado los cuadernos *SQLite Básico*, *Intermedio* y *Avanzado* |
| **Qué producirás** | El diagrama ER de tu proyecto, normalizado hasta 3FN, e implementado en SQLite con datos de prueba |
| **Tiempo estimado** | 100 minutos |

## Cómo trabajar este cuaderno

1. Ejecuta las celdas **en orden**, de arriba hacia abajo.
2. Las celdas de este cuaderno **crean tablas reales** para que veas cada idea funcionando. Se pueden ejecutar varias veces sin errores.
3. Al final está el **Lab D4**: un ejemplo de referencia completo y la lista de entrega de tu proyecto.

## 1. ¿Qué es el modelo relacional?

El modelo relacional es la base teórica de los sistemas de bases de datos que usan tablas. Lo propuso **Edgar F. Codd** en 1970, en IBM. La idea central: organizar los datos en **tablas (relaciones)** formadas por filas y columnas, **conectadas mediante claves**.

| Concepto | En el modelo relacional | En SQLite (práctica) |
|---|---|---|
| Relación | Tabla con filas y columnas | `CREATE TABLE empleados (...)` |
| Tupla | Una fila de la tabla | `INSERT INTO empleados VALUES (...)` |
| Atributo | Una columna de la tabla | `nombre TEXT NOT NULL` |
| Dominio | Tipo de dato permitido | `INTEGER`, `TEXT`, `REAL`... |
| Clave primaria | Identifica cada fila de forma única | `id INTEGER PRIMARY KEY` |
| Clave foránea | Conecta dos tablas | `FOREIGN KEY (id_depto) REFERENCES departamentos(id_depto)` |
| Integridad referencial | Una clave foránea debe apuntar a una clave primaria que **exista** | La garantiza `FOREIGN KEY`, **si activas** `PRAGMA foreign_keys = ON` |

> **¿Por qué aprender el modelo relacional?** Antes de escribir consultas hay que saber **qué se va a construir**. El diseño decide qué consultas serán fáciles o imposibles. Un buen modelo significa menos código, menos errores y más velocidad.

### Preparación del entorno

Ejecuta la celda de preparación: carga `sqlite3` y `pandas`, abre la conexión `conn` y **activa las claves foráneas**. Las sentencias que cambian la base se escriben con `conn.executescript(...)` y las consultas con `pd.read_sql_query(..., conn)`.

In [1]:
# CELDA DE PREPARACIÓN: ejecútala una sola vez al inicio.
import sqlite3
import pandas as pd

conn = sqlite3.connect("modelado.db")      # crea (o abre) el archivo de la base de datos
conn.execute("PRAGMA foreign_keys = ON")   # SQLite necesita esta línea para respetar las claves foráneas

print("Versión de SQLite:", sqlite3.sqlite_version)

Versión de SQLite: 3.45.1


## 2. Modelo Entidad-Relación (ER)

El modelo ER lo creó **Peter Chen** en 1976. Es la herramienta estándar para **diseñar** una base de datos antes de implementarla. Tiene tres elementos: **entidades**, **atributos** y **relaciones**.

### 2.1 Entidades

| Entidad | ¿Qué representa? | Tabla en SQLite |
|---|---|---|
| `CLIENTE` | Una persona o empresa que compra | `CREATE TABLE clientes (...)` |
| `PRODUCTO` | Un artículo que se vende | `CREATE TABLE productos (...)` |
| `EMPLEADO` | Una persona que trabaja en la empresa | `CREATE TABLE empleados (...)` |
| `PEDIDO` | Una transacción de compra | `CREATE TABLE pedidos (...)` |
| `DEPARTAMENTO` | Un área de la empresa | `CREATE TABLE departamentos (...)` |

### 2.2 Atributos

| Tipo de atributo | Descripción | Ejemplo |
|---|---|---|
| Clave (PK) | Identifica de forma única cada fila | `id_cliente` (subrayado en el diagrama) |
| Simple | Un solo valor, indivisible | `nombre`, `precio`, `fecha` |
| Compuesto | Se puede dividir en partes | `dirección` = calle + número + ciudad |
| Multivalorado | Puede tener varios valores | `teléfonos` (varios por cliente) |
| Derivado | Se calcula a partir de otro | `edad` (se calcula de `fecha_nacimiento`) |
| Nulo (`NULL`) | Puede no tener valor | `email` (opcional) |

### 2.3 Relaciones: diagrama ER del sistema de ventas

```
      1                      N         N                       M
CLIENTE ────── realiza ──────► PEDIDO ◄────── contiene ──────► PRODUCTO
id_cliente                     id_pedido                       id_producto
nombre, ciudad                 fecha                           nombre, precio
```

**Cómo se lee:** un cliente realiza **muchos** pedidos, pero cada pedido pertenece a **un** cliente (**1:N**). Un pedido contiene **muchos** productos y un producto aparece en **muchos** pedidos (**N:M**).

Rectángulos = entidades · Óvalos = atributos · Rombos = relaciones · Los números (1, N, M) indican la **cardinalidad** de cada extremo.

## 3. Cardinalidad: 1:1, 1:N y N:M

La **cardinalidad** indica cuántas instancias de una entidad pueden relacionarse con cuántas de otra. Es la decisión de diseño más importante, porque determina **cómo se implementan las tablas**.

### 3.1 Relación 1:1 (uno a uno)

| Ejemplo real | Entidad A | Entidad B | Interpretación |
|---|---|---|---|
| Persona ↔ Pasaporte | `PERSONA` | `PASAPORTE` | Cada persona tiene un solo pasaporte |
| País ↔ Capital | `PAIS` | `CIUDAD_CAPITAL` | Cada país tiene una sola capital |
| Empleado ↔ Credencial | `EMPLEADO` | `CREDENCIAL` | Cada empleado tiene una sola credencial |

Se implementa con una clave foránea **más `UNIQUE`** en la tabla secundaria: `UNIQUE` impide que dos pasaportes apunten a la misma persona.

In [2]:
conn.executescript("""
DROP TABLE IF EXISTS pasaportes;
DROP TABLE IF EXISTS personas;

CREATE TABLE personas (
    id_persona INTEGER PRIMARY KEY,
    nombre     TEXT NOT NULL
);

CREATE TABLE pasaportes (
    id_pasaporte INTEGER PRIMARY KEY,
    codigo       TEXT UNIQUE NOT NULL,
    id_persona   INTEGER UNIQUE NOT NULL,        -- UNIQUE asegura la relación 1:1
    FOREIGN KEY (id_persona) REFERENCES personas(id_persona)
);

INSERT INTO personas   VALUES (1, 'Rosa Huanca'), (2, 'Luis Ttito');
INSERT INTO pasaportes VALUES (1, 'P-0001', 1);
""")

**¿Y si intentamos dar un segundo pasaporte a la persona 1?**

```sql
INSERT INTO pasaportes VALUES (2, 'P-0002', 1);
```

SQLite lo **rechaza** y responde `UNIQUE constraint failed: pasaportes.id_persona`: gracias a `UNIQUE`, cada persona solo puede tener un pasaporte. (No ejecutes esta sentencia: detendría la ejecución del cuaderno.)

### 3.2 Relación 1:N (uno a muchos)

**La clave foránea siempre va en el lado N** (el lado «muchos»).

| Ejemplo real | Lado 1 (padre) | Lado N (hijo) | La FK va en... |
|---|---|---|---|
| Departamento → Empleados | `DEPARTAMENTO` | `EMPLEADO` | `EMPLEADO` |
| Cliente → Pedidos | `CLIENTE` | `PEDIDO` | `PEDIDO` |
| Categoría → Productos | `CATEGORIA` | `PRODUCTO` | `PRODUCTO` |
| Profesor → Cursos | `PROFESOR` | `CURSO` | `CURSO` |

Con `PRAGMA foreign_keys = ON`, SQLite **rechaza** un empleado de un departamento que no existe.

In [3]:
conn.executescript("""
DROP TABLE IF EXISTS empleados;
DROP TABLE IF EXISTS departamentos;

CREATE TABLE departamentos (                 -- tabla PADRE (lado 1)
    id_depto INTEGER PRIMARY KEY,
    nombre   TEXT NOT NULL UNIQUE
);

CREATE TABLE empleados (                     -- tabla HIJA (lado N)
    id_emp   INTEGER PRIMARY KEY,
    nombre   TEXT NOT NULL,
    id_depto INTEGER NOT NULL,               -- la FK apunta al padre
    FOREIGN KEY (id_depto) REFERENCES departamentos(id_depto)
);

INSERT INTO departamentos VALUES (1, 'Sistemas'), (2, 'Ventas');
INSERT INTO empleados     VALUES (1, 'Carlos Quispe', 1), (2, 'Ana Huanca', 2);
""")

**¿Y si intentamos crear un empleado del departamento 99, que no existe?**

```sql
INSERT INTO empleados VALUES (3, 'Persona Fantasma', 99);
```

SQLite lo **rechaza** y responde `FOREIGN KEY constraint failed`: la clave foránea impide que un empleado apunte a un departamento inexistente. Esto es la **integridad referencial** en acción. (No ejecutes esta sentencia: detendría la ejecución del cuaderno.)

### 3.3 Relación N:M (muchos a muchos)

Se resuelve creando una **tabla intermedia** (tabla pivote) con las claves foráneas de ambas tablas.

| Ejemplo real | Entidad A | Entidad B | Tabla intermedia |
|---|---|---|---|
| Estudiantes ↔ Cursos | `ESTUDIANTE` | `CURSO` | `matriculas` (id_estudiante, id_curso) |
| Pedidos ↔ Productos | `PEDIDO` | `PRODUCTO` | `detalle_pedido` (id_pedido, id_producto) |
| Médicos ↔ Pacientes | `MEDICO` | `PACIENTE` | `consultas` (id_medico, id_paciente) |
| Actores ↔ Películas | `ACTOR` | `PELICULA` | `actuaciones` (id_actor, id_pelicula) |

> **Atributos de la relación N:M:** la tabla intermedia puede tener atributos propios que pertenecen a la **relación** y no a ninguna de las dos entidades. Ejemplo: `cantidad` y `precio_unitario` pertenecen a la relación entre *ese* pedido y *ese* producto.

In [4]:
conn.executescript("""
DROP TABLE IF EXISTS detalle_pedido;
DROP TABLE IF EXISTS pedidos;
DROP TABLE IF EXISTS productos;

CREATE TABLE pedidos (
    id_pedido INTEGER PRIMARY KEY,
    fecha     TEXT NOT NULL
);

CREATE TABLE productos (
    id_producto INTEGER PRIMARY KEY,
    nombre      TEXT NOT NULL,
    precio      REAL NOT NULL
);

CREATE TABLE detalle_pedido (                -- tabla INTERMEDIA (relación N:M)
    id_pedido       INTEGER NOT NULL,        -- FK → pedidos
    id_producto     INTEGER NOT NULL,        -- FK → productos
    cantidad        INTEGER NOT NULL DEFAULT 1,
    precio_unitario REAL    NOT NULL,        -- precio EN EL MOMENTO de la compra
    PRIMARY KEY (id_pedido, id_producto),    -- clave primaria COMPUESTA
    FOREIGN KEY (id_pedido)   REFERENCES pedidos(id_pedido),
    FOREIGN KEY (id_producto) REFERENCES productos(id_producto)
);

INSERT INTO pedidos   VALUES (1, '2024-03-01'), (2, '2024-03-02');
INSERT INTO productos VALUES (1, 'Licencia Office 365', 450.0), (2, 'Antivirus Anual', 180.0);
INSERT INTO detalle_pedido VALUES (1, 1, 2, 450.0), (1, 2, 1, 180.0), (2, 2, 3, 180.0);
""")

print("Un pedido con varios productos, y un producto en varios pedidos:")
pd.read_sql_query("""
SELECT d.id_pedido AS Pedido, pr.nombre AS Producto, d.cantidad AS Cantidad,
       d.cantidad * d.precio_unitario AS Subtotal
FROM   detalle_pedido d
INNER JOIN productos pr ON d.id_producto = pr.id_producto
ORDER BY d.id_pedido, pr.nombre
""", conn)

Un pedido con varios productos, y un producto en varios pedidos:


,Pedido,Producto,Cantidad,Subtotal
0,1,Antivirus Anual,1,180.0
1,1,Licencia Office 365,2,900.0
2,2,Antivirus Anual,3,540.0


## 4. Normalización: 1FN, 2FN y 3FN

La normalización elimina la **redundancia** (datos repetidos) y las **dependencias problemáticas**. Cada forma normal es una regla más estricta que la anterior.

| Forma normal | Regla que debe cumplir | ¿Qué elimina? |
|---|---|---|
| **1FN** | Un solo valor por celda. Sin grupos repetidos. Tiene clave primaria | Celdas con varios valores |
| **2FN** | Cumple 1FN y cada atributo depende de **toda** la clave primaria | Dependencias parciales de la clave |
| **3FN** | Cumple 2FN y ningún atributo depende de **otro atributo que no es clave** | Dependencias transitivas |
| BCNF | Cada determinante es clave candidata (nivel avanzado) | Anomalías residuales de la 3FN |

### 4.1 Primera Forma Normal (1FN)

Cada celda tiene **un solo valor atómico**, no hay grupos repetidos y existe una clave primaria. La tabla siguiente **viola la 1FN**: `telefonos` y `productos_comprados` guardan varios valores en una celda.

In [5]:
violacion_1fn = pd.DataFrame({
    "id_cliente": [1, 2],
    "nombre": ["María Quispe", "Carlos Ttito"],
    "telefonos": ["984111222, 984333444", "984555666"],
    "productos_comprados": ["Office, Antivirus, Soporte", "Laptop, Disco"],
})
violacion_1fn

,id_cliente,nombre,telefonos,productos_comprados
0,1,María Quispe,"984111222, 984333444","Office, Antivirus, Soporte"
1,2,Carlos Ttito,984555666,"Laptop, Disco"


**La solución:** cada teléfono pasa a una fila de una **tabla separada** (relación 1:N con el cliente). Lo mismo se haría con los productos comprados (relación N:M, como en 3.3).

In [6]:
conn.executescript("""
DROP TABLE IF EXISTS telefonos;
DROP TABLE IF EXISTS clientes_1fn;

CREATE TABLE clientes_1fn (                  -- tabla clientes normalizada (1FN)
    id_cliente INTEGER PRIMARY KEY,
    nombre     TEXT NOT NULL
);

CREATE TABLE telefonos (                     -- los teléfonos en tabla separada
    id_tel     INTEGER PRIMARY KEY,
    numero     TEXT NOT NULL,
    id_cliente INTEGER NOT NULL,
    FOREIGN KEY (id_cliente) REFERENCES clientes_1fn(id_cliente)
);

INSERT INTO clientes_1fn VALUES (1, 'María Quispe'), (2, 'Carlos Ttito');
INSERT INTO telefonos (numero, id_cliente) VALUES
    ('984111222', 1), ('984333444', 1), ('984555666', 2);
""")
pd.read_sql_query("""
SELECT c.nombre AS Cliente, t.numero AS Telefono
FROM   clientes_1fn c
INNER JOIN telefonos t ON c.id_cliente = t.id_cliente
ORDER BY c.nombre, t.numero
""", conn)

,Cliente,Telefono
0,Carlos Ttito,984555666
1,María Quispe,984111222
2,María Quispe,984333444


### 4.2 Segunda Forma Normal (2FN)

Cumple 1FN y **todos los atributos que no son clave dependen de la clave primaria COMPLETA**. Solo aplica cuando la clave primaria es **compuesta** (formada por varias columnas).

La tabla siguiente **viola la 2FN**: la clave es `(id_pedido, id_producto)`, pero `nombre_producto` depende **solo** de `id_producto` (una parte de la clave).

In [7]:
violacion_2fn = pd.DataFrame({
    "id_pedido":       [1, 1, 2],
    "id_producto":     [1, 2, 2],
    "cantidad":        [2, 1, 3],
    "nombre_producto": ["Licencia Office 365", "Antivirus Anual", "Antivirus Anual"],   # se repite
})
violacion_2fn

,id_pedido,id_producto,cantidad,nombre_producto
0,1,1,2,Licencia Office 365
1,1,2,1,Antivirus Anual
2,2,2,3,Antivirus Anual


**La solución** es la que ya construimos en 3.3: `detalle_pedido` guarda **solo lo que depende de la pareja** (pedido + producto): `cantidad` y `precio_unitario`; y `productos` guarda **lo que depende del producto**: `nombre` y `precio`. Así el nombre del producto se escribe una sola vez.

### 4.3 Tercera Forma Normal (3FN)

Cumple 2FN y **ningún atributo no clave depende de otro atributo no clave** (sin dependencias transitivas). Ejemplo: en una tabla de trabajadores, `ciudad` depende de `cod_postal`, no de `id_trabajador`. Si se repite la ciudad en cada fila, se corrige creando una tabla `ciudades`.

> **Orden de creación:** la tabla **padre** (`ciudades`) se crea **antes** que la tabla que la referencia (`trabajadores`).

In [8]:
conn.executescript("""
DROP TABLE IF EXISTS trabajadores;
DROP TABLE IF EXISTS ciudades;

CREATE TABLE ciudades (                      -- 1.º: la tabla padre. Relación: código postal → nombre
    cod_postal TEXT PRIMARY KEY,
    nombre     TEXT NOT NULL                 -- ahora el nombre está en UN solo lugar
);

CREATE TABLE trabajadores (                  -- 2.º: la tabla hija. Solo datos del trabajador
    id_trabajador INTEGER PRIMARY KEY,
    nombre        TEXT NOT NULL,
    cod_postal    TEXT NOT NULL,             -- solo el código postal
    FOREIGN KEY (cod_postal) REFERENCES ciudades(cod_postal)
);

INSERT INTO ciudades     VALUES ('08000', 'Cusco'), ('21001', 'Puno');
INSERT INTO trabajadores VALUES (1, 'Carlos Quispe', '08000'), (2, 'Ana Huanca', '21001'), (3, 'Maria Ccopa', '08000');
""")
pd.read_sql_query("""
SELECT t.nombre AS Trabajador, c.nombre AS Ciudad
FROM   trabajadores t
INNER JOIN ciudades c ON t.cod_postal = c.cod_postal
ORDER BY t.nombre
""", conn)

,Trabajador,Ciudad
0,Ana Huanca,Puno
1,Carlos Quispe,Cusco
2,Maria Ccopa,Cusco


## 5. De diagrama ER a tablas SQLite

### Reglas de mapeo ER → tablas

1. Cada **ENTIDAD** → una tabla (nombre en plural).
2. Cada **ATRIBUTO** → una columna con su tipo.
3. Atributo **CLAVE** → `PRIMARY KEY` (+ `AUTOINCREMENT` si es numérico).
4. Relación **1:N** → `FOREIGN KEY` en la tabla del lado N.
5. Relación **N:M** → tabla intermedia con las dos claves foráneas.
6. Relación **1:1** → `FOREIGN KEY` + `UNIQUE` en la tabla secundaria.
7. Atributo **MULTIVALORADO** → tabla separada con clave foránea.

### Caso completo: Sistema de Matrícula Universitaria

**Entidades:** ESTUDIANTE, CURSO, PROFESOR.
**Relaciones:** ESTUDIANTE **N:M** CURSO (mediante MATRICULA) · PROFESOR **1:N** CURSO.

```
PROFESOR 1 ──── dicta ────► N CURSO N ◄──── se matricula ────► M ESTUDIANTE
                                         (tabla intermedia: MATRICULA)
```

El orden de creación es: **1) profesores y estudiantes** (no tienen clave foránea) → **2) cursos** (apunta a profesores) → **3) matrículas** (apunta a estudiantes y cursos).

In [9]:
conn.executescript("""
DROP TABLE IF EXISTS matriculas;
DROP TABLE IF EXISTS estudiantes;
DROP TABLE IF EXISTS cursos;
DROP TABLE IF EXISTS profesores;

-- 1. PROFESORES (sin FK, va primero)
CREATE TABLE profesores (
    id_profesor  INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre       TEXT NOT NULL,
    especialidad TEXT NOT NULL,
    email        TEXT UNIQUE
);

-- 2. CURSOS (FK → profesores: relación 1:N)
CREATE TABLE cursos (
    id_curso    INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre      TEXT NOT NULL,
    creditos    INTEGER NOT NULL DEFAULT 4,
    id_profesor INTEGER NOT NULL,                -- FK → profesores (lado N)
    FOREIGN KEY (id_profesor) REFERENCES profesores(id_profesor)
);

-- 3. ESTUDIANTES (sin FK)
CREATE TABLE estudiantes (
    id_estudiante INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre        TEXT NOT NULL,
    codigo        TEXT UNIQUE NOT NULL,
    ciudad        TEXT DEFAULT 'Cusco'
);

-- 4. MATRICULAS (tabla intermedia: relación N:M)
CREATE TABLE matriculas (
    id_matricula    INTEGER PRIMARY KEY AUTOINCREMENT,
    id_estudiante   INTEGER NOT NULL,
    id_curso        INTEGER NOT NULL,
    fecha_matricula TEXT NOT NULL,
    nota_final      REAL DEFAULT NULL,
    FOREIGN KEY (id_estudiante) REFERENCES estudiantes(id_estudiante),
    FOREIGN KEY (id_curso)      REFERENCES cursos(id_curso),
    UNIQUE (id_estudiante, id_curso)             -- un estudiante no se matricula dos veces en el mismo curso
);
""")
print("Base de datos de la universidad creada")

Base de datos de la universidad creada


### Insertar datos de prueba

In [10]:
conn.executescript("""
INSERT INTO profesores (nombre, especialidad, email) VALUES
    ('Pilar Rocio Sayan Mejia', 'Data Science',   'psayan@tecsup.edu.pe'),   -- id = 1
    ('Carlos Mamani',           'Bases de Datos', 'cmamani@tecsup.edu.pe'),  -- id = 2
    ('Ana Quispe',              'Programacion',   'aquispe@tecsup.edu.pe');  -- id = 3

INSERT INTO cursos (nombre, creditos, id_profesor) VALUES
    ('Fundamentos de Gestion de Datos', 4, 1),   -- id = 1
    ('Mineria de Datos',                4, 1),   -- id = 2
    ('SQLite Avanzado',                 3, 2),   -- id = 3
    ('Python para Datos',               4, 3);   -- id = 4

INSERT INTO estudiantes (nombre, codigo, ciudad) VALUES
    ('Luis Ttito',  '2024001', 'Cusco'),   -- id = 1
    ('Rosa Huanca', '2024002', 'Cusco'),   -- id = 2
    ('Marco Apaza', '2024003', 'Puno'),    -- id = 3
    ('Diana Flores','2024004', 'Cusco');   -- id = 4

INSERT INTO matriculas (id_estudiante, id_curso, fecha_matricula) VALUES
    (1, 1, '2024-03-01'),   -- Luis  → Fundamentos
    (1, 3, '2024-03-01'),   -- Luis  → SQLite Avanzado
    (2, 1, '2024-03-02'),   -- Rosa  → Fundamentos
    (4, 2, '2024-03-04'),   -- Diana → Minería
    (4, 3, '2024-03-04');   -- Diana → SQLite Avanzado
""")
print("Datos de prueba insertados")

Datos de prueba insertados


### Consultar el modelo completo

Un `JOIN` de cuatro tablas recorre todo el modelo: **matrícula → estudiante → curso → profesor**.

In [11]:
pd.read_sql_query("""
SELECT e.nombre          AS Estudiante,
       c.nombre          AS Curso,
       p.nombre          AS Profesor,
       m.fecha_matricula AS Matriculado
FROM   matriculas m
INNER JOIN estudiantes e ON m.id_estudiante = e.id_estudiante
INNER JOIN cursos      c ON m.id_curso      = c.id_curso
INNER JOIN profesores  p ON c.id_profesor   = p.id_profesor
ORDER BY e.nombre, c.nombre
""", conn)

,Estudiante,Curso,Profesor,Matriculado
0,Diana Flores,Mineria de Datos,Pilar Rocio Sayan Mejia,2024-03-04
1,Diana Flores,SQLite Avanzado,Carlos Mamani,2024-03-04
2,Luis Ttito,Fundamentos de Gestion de Datos,Pilar Rocio Sayan Mejia,2024-03-01
3,Luis Ttito,SQLite Avanzado,Carlos Mamani,2024-03-01
4,Rosa Huanca,Fundamentos de Gestion de Datos,Pilar Rocio Sayan Mejia,2024-03-02


### El modelo se protege solo

Gracias a las claves foráneas y a `UNIQUE`, la base **rechaza datos incoherentes**. Estos son dos intentos que SQLite no permite:

**1) Matricular a un estudiante que no existe (id 99):**

```sql
INSERT INTO matriculas (id_estudiante, id_curso, fecha_matricula) VALUES (99, 1, '2024-03-05');
```

SQLite responde `FOREIGN KEY constraint failed`.

**2) Matricular dos veces al mismo estudiante en el mismo curso:**

```sql
INSERT INTO matriculas (id_estudiante, id_curso, fecha_matricula) VALUES (1, 1, '2024-03-05');
```

SQLite responde `UNIQUE constraint failed: matriculas.id_estudiante, matriculas.id_curso`.

> No ejecutes estas sentencias en el cuaderno: cada una produce un error y detendría la ejecución. Pruébalas solo si quieres ver el mensaje.

## 6. Lab D4: Diseño ER del proyecto grupal

> **Objetivo:** el equipo diseña el diagrama ER de su proyecto, lo **normaliza hasta 3FN** e implementa el modelo en SQLite desde Google Colab con **datos de prueba reales**. Al finalizar, cada grupo tendrá una base de datos funcional en su cuaderno.

### Paso 1. Identificar entidades, atributos y relaciones

| Pregunta guía | Ejemplo (proyecto de detección de fraude con tarjetas) |
|---|---|
| ¿Qué objetos queremos registrar? | Cliente, Tarjeta, Transacción, Comercio |
| ¿Qué datos guardamos de cada objeto? | Cliente: DNI, nombre, correo, ciudad |
| ¿Cómo se relacionan los objetos? | Un cliente **tiene** muchas tarjetas (1:N) |
| ¿Cuál es la cardinalidad de cada relación? | Tarjeta **N:M** Comercio (mediante Transacción) |
| ¿Qué atributo identifica cada entidad? | `id_cliente`, `id_tarjeta`, `id_transaccion` |
| ¿Qué atributos son obligatorios (`NOT NULL`)? | DNI, nombre, número de tarjeta |
| ¿Qué atributos son únicos (`UNIQUE`)? | Correo del cliente, número de tarjeta |

### Paso 2. Verificar la normalización (lista de control)

- **1FN:** ¿cada celda tiene un solo valor? Si guardas varios correos o teléfonos en un campo, sepáralos en otra tabla.
- **2FN:** ¿todos los atributos dependen de la clave primaria completa? Si la clave es compuesta y un atributo depende de solo una parte, muévelo a otra tabla.
- **3FN:** ¿hay atributos no clave que dependen de otro atributo no clave? Ejemplo: `ciudad` depende de `cod_postal`, no de `id_cliente`. Si es así, crea la tabla `ciudades(cod_postal, nombre)`.

### Paso 3. Ejemplo de referencia (cuatro tablas, datos ficticios)

Este ejemplo **funciona de principio a fin**: úsalo como plantilla para tu proyecto, cambiando entidades y datos por los de tu equipo. Reglas que debe cumplir tu base: **mínimo 5 filas por tabla** y **datos coherentes con el contexto del proyecto** (nada de «nombre1» o «test»).

> Todos los nombres y documentos de este ejemplo son **ficticios**. No uses datos personales reales de nadie en tu proyecto.

In [12]:
conn.executescript("""
DROP TABLE IF EXISTS transacciones;
DROP TABLE IF EXISTS tarjetas;
DROP TABLE IF EXISTS comercios;
DROP TABLE IF EXISTS clientes;

CREATE TABLE clientes (                          -- entidad sin FK: va primero
    id_cliente INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre     TEXT NOT NULL,
    dni        TEXT UNIQUE NOT NULL,
    email      TEXT UNIQUE,
    ciudad     TEXT DEFAULT 'Cusco'
);

CREATE TABLE comercios (                         -- entidad sin FK
    id_comercio INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre      TEXT NOT NULL,
    rubro       TEXT NOT NULL
);

CREATE TABLE tarjetas (                          -- 1:N con clientes
    id_tarjeta INTEGER PRIMARY KEY AUTOINCREMENT,
    numero     TEXT UNIQUE NOT NULL,
    id_cliente INTEGER NOT NULL,
    FOREIGN KEY (id_cliente) REFERENCES clientes(id_cliente)
);

CREATE TABLE transacciones (                     -- tabla intermedia: tarjetas N:M comercios
    id_transaccion INTEGER PRIMARY KEY AUTOINCREMENT,
    fecha          TEXT NOT NULL,
    monto          REAL NOT NULL,
    id_tarjeta     INTEGER NOT NULL,
    id_comercio    INTEGER NOT NULL,
    FOREIGN KEY (id_tarjeta)  REFERENCES tarjetas(id_tarjeta),
    FOREIGN KEY (id_comercio) REFERENCES comercios(id_comercio)
);
""")
print("Tablas del proyecto de ejemplo creadas")

Tablas del proyecto de ejemplo creadas


### Paso 4. Insertar datos de prueba (mínimo 5 filas por tabla)

In [13]:
conn.executescript("""
INSERT INTO clientes (nombre, dni, email, ciudad) VALUES
    ('Maria Quispe Ttito', '70000001', 'mquispe@correo.pe',  'Cusco'),
    ('Carlos Huanca Ruiz', '70000002', 'chuanca@correo.pe',  'Cusco'),
    ('Ana Mamani Cruz',    '70000003', 'amamani@correo.pe',  'Puno'),
    ('Pedro Condori Paz',  '70000004', NULL,                 'Arequipa'),
    ('Lucia Apaza Rojas',  '70000005', 'lapaza@correo.pe',   'Lima');

INSERT INTO comercios (nombre, rubro) VALUES
    ('Supermercado Ttio', 'Alimentos'),
    ('Grifo El Sol',      'Combustible'),
    ('Tienda TecnoCusco', 'Tecnologia'),
    ('Restaurante Inti',  'Restaurantes'),
    ('Farmacia Salud',    'Salud');

INSERT INTO tarjetas (numero, id_cliente) VALUES
    ('0000-0000-0001', 1),
    ('0000-0000-0002', 1),
    ('0000-0000-0003', 2),
    ('0000-0000-0004', 3),
    ('0000-0000-0005', 5);

INSERT INTO transacciones (fecha, monto, id_tarjeta, id_comercio) VALUES
    ('2024-05-01',   85.50, 1, 1),
    ('2024-05-02',  120.00, 1, 2),
    ('2024-05-02', 3499.00, 3, 3),
    ('2024-05-03',   64.90, 4, 4),
    ('2024-05-04',   45.00, 5, 5),
    ('2024-05-05',  210.30, 2, 1),
    ('2024-05-06', 1899.00, 3, 3),
    ('2024-05-07',   38.00, 4, 5);
""")
print("Datos de prueba insertados")

Datos de prueba insertados


### Paso 5. Consulta de verificación (con `JOIN` que cruce las tablas principales)

Recorre todo el modelo: **transacción → tarjeta → cliente** y **transacción → comercio**. Las transacciones más grandes son las primeras que un equipo antifraude revisaría.

In [14]:
pd.read_sql_query("""
SELECT t.fecha        AS Fecha,
       c.nombre       AS Cliente,
       tj.numero      AS Tarjeta,
       co.nombre      AS Comercio,
       co.rubro       AS Rubro,
       t.monto        AS Monto
FROM   transacciones t
INNER JOIN tarjetas  tj ON t.id_tarjeta  = tj.id_tarjeta
INNER JOIN clientes  c  ON tj.id_cliente = c.id_cliente
INNER JOIN comercios co ON t.id_comercio = co.id_comercio
ORDER BY t.monto DESC
""", conn)

,Fecha,Cliente,Tarjeta,Comercio,Rubro,Monto
0,2024-05-02,Carlos Huanca Ruiz,0000-0000-0003,Tienda TecnoCusco,Tecnologia,3499.0
1,2024-05-06,Carlos Huanca Ruiz,0000-0000-0003,Tienda TecnoCusco,Tecnologia,1899.0
2,2024-05-05,Maria Quispe Ttito,0000-0000-0002,Supermercado Ttio,Alimentos,210.3
3,2024-05-02,Maria Quispe Ttito,0000-0000-0001,Grifo El Sol,Combustible,120.0
4,2024-05-01,Maria Quispe Ttito,0000-0000-0001,Supermercado Ttio,Alimentos,85.5
5,2024-05-03,Ana Mamani Cruz,0000-0000-0004,Restaurante Inti,Restaurantes,64.9
6,2024-05-04,Lucia Apaza Rojas,0000-0000-0005,Farmacia Salud,Salud,45.0
7,2024-05-07,Ana Mamani Cruz,0000-0000-0004,Farmacia Salud,Salud,38.0


## 7. Lista de entrega del Lab D4

| N.° | Entregable | Qué debe incluir | Puntos |
|---|---|---|---|
| 1 | **Diagrama ER completo** (Lucidchart, draw.io o papel fotografiado) | Entidades, atributos, relaciones y cardinalidades claramente indicadas | 25 |
| 2 | **Documento de normalización** | Justificación de 1FN, 2FN y 3FN aplicadas al modelo | 20 |
| 3 | **Cuaderno de Google Colab (.ipynb)** | Celda de conexión, `CREATE TABLE` comentado e `INSERT` de datos de prueba | 30 |
| 4 | **Consulta de verificación** | Al menos un `SELECT` con `JOIN` que cruce las tablas principales | 15 |
| 5 | **Presentación en clase** (5 minutos por grupo) | Explicar el modelo ER y demostrar el Colab funcionando | 10 |
| | **Total** | | **100** |

### Errores frecuentes que cuestan puntos

- Crear el cuaderno pero **no ejecutarlo**: las celdas deben mostrar sus resultados.
- Datos de prueba genéricos («nombre1», «test»): deben ser datos del contexto real del proyecto.
- Clave foránea sin la tabla padre creada primero: **el orden de los `CREATE TABLE` importa**.
- No indicar las cardinalidades en el diagrama ER (1:N, N:M).
- Mezclar atributos de distintas entidades en una sola tabla (viola 2FN o 3FN).
- Olvidar `PRAGMA foreign_keys = ON`: sin esa línea, SQLite **no verifica** las claves foráneas.

### Consejos para un Lab D4 excelente

- Usa nombres descriptivos: `id_cliente`, `fecha_pedido` (no `id`, `f1`).
- Comenta cada línea de `CREATE TABLE` explicando qué hace.
- Verifica con `SELECT * FROM tabla` que los datos se insertaron bien.
- El diagrama ER debe coincidir **exactamente** con las tablas que creas en Colab.

## 8. Referencias bibliográficas recomendadas

| Libro | Autor(es) | Editorial | Temas clave |
|---|---|---|---|
| *SQL and Relational Theory* (3.ª ed.) | C. J. Date | O'Reilly | Teoría relacional, normalización |
| *Database Administration* (2.ª ed.) | Craig S. Mullins | Addison-Wesley | Diseño de esquemas, normalización práctica |
| *Learning SQL* (3.ª ed.) | Alan Beaulieu | O'Reilly | `SELECT`, `JOIN`, `GROUP BY` desde cero |
| *Database Design for Mere Mortals* | Michael J. Hernandez | Addison-Wesley | Diseño ER accesible, cardinalidad, tablas de unión |
| *Using SQLite* | Jay A. Kreibich | O'Reilly | Guía de SQLite |

**Acceso a los libros:** O'Reilly Learning (learning.oreilly.com, con prueba gratuita), Google Books (vista previa de capítulos), Internet Archive (préstamo digital) y la biblioteca digital institucional.

## 9. Ejercicios

Resuélvelos en papel o en celdas nuevas. La solución está escondida.

---

**Ejercicio 1. Cardinalidades.** Indica si cada pareja es **1:1**, **1:N** o **N:M**: (a) Persona – Pasaporte, (b) Madre – Hijos, (c) Autores – Libros, (d) Cliente – Pedidos, (e) Estudiantes – Cursos.

<details><summary>Ver solución</summary>

(a) **1:1** · (b) **1:N** · (c) **N:M** (un autor escribe varios libros y un libro puede tener varios autores) · (d) **1:N** · (e) **N:M**
</details>

---

**Ejercicio 2. Primera forma normal.** La tabla `alumnos(id, nombre, cursos)` guarda en `cursos` valores como `'Python, SQLite'`. Rediséñala para cumplir la 1FN y escribe las tablas.

<details><summary>Ver solución</summary>

Es una relación **N:M** (un alumno lleva varios cursos y un curso tiene varios alumnos): se separa en tres tablas.

```python
conn.executescript("""
CREATE TABLE alumnos   (id_alumno INTEGER PRIMARY KEY, nombre TEXT NOT NULL);
CREATE TABLE cursos_e2 (id_curso  INTEGER PRIMARY KEY, nombre TEXT NOT NULL);
CREATE TABLE inscripciones (
    id_alumno INTEGER NOT NULL,
    id_curso  INTEGER NOT NULL,
    PRIMARY KEY (id_alumno, id_curso),
    FOREIGN KEY (id_alumno) REFERENCES alumnos(id_alumno),
    FOREIGN KEY (id_curso)  REFERENCES cursos_e2(id_curso)
);
""")
```
</details>

---

**Ejercicio 3. Tercera forma normal.** La tabla `ventas(id_venta, id_cliente, nombre_cliente, ciudad_cliente, monto)` repite el nombre y la ciudad del cliente en cada venta. ¿Qué forma normal viola y cómo se corrige?

<details><summary>Ver solución</summary>

Viola la **3FN**: `nombre_cliente` y `ciudad_cliente` dependen de `id_cliente`, que **no es la clave** de `ventas` (la clave es `id_venta`). Se corrige creando la tabla `clientes` (con `id_cliente`, `nombre` y `ciudad`) y dejando en `ventas` solo `id_cliente` como clave foránea.

```python
conn.executescript("""
CREATE TABLE clientes_e3 (id_cliente INTEGER PRIMARY KEY, nombre TEXT NOT NULL, ciudad TEXT);
CREATE TABLE ventas_e3 (
    id_venta   INTEGER PRIMARY KEY,
    id_cliente INTEGER NOT NULL,
    monto      REAL NOT NULL,
    FOREIGN KEY (id_cliente) REFERENCES clientes_e3(id_cliente)
);
""")
```
</details>

---

## 10. Resumen

| Idea | Lo que debes recordar |
|---|---|
| Modelo ER | Entidades, atributos y relaciones: se **diseña antes** de implementar |
| Cardinalidad 1:N | La clave foránea va en la tabla del lado **N** |
| Cardinalidad N:M | Se resuelve con una **tabla intermedia** con las dos claves foráneas |
| Cardinalidad 1:1 | Clave foránea **más** `UNIQUE` |
| Normalización | 1FN (un valor por celda), 2FN (depende de toda la clave), 3FN (sin dependencias transitivas) |
| Orden de creación | Primero las tablas **padre**, luego las **hijas** |
| Integridad referencial | Solo se cumple con `PRAGMA foreign_keys = ON` |

---

*Modelado Relacional con SQLite · TECSUP · Fundamentos de Gestión de Datos · Docente: Pilar Rocío Sayán Mejía*